# Forecast regional time series

Regional-mean daily PRECT and FLUT for every hindcast member, the ensemble mean and 25th-75th-percentile envelope, aligned with daily observations, plotted against forecast date.

Cases: **S0** hindcasts start from the Dec 2011 analyses and cover Jan-Feb 2012; **S1** hindcasts start from the May 2012 analyses and cover Jun-Jul 2012. Each case compares hindcasts initialized from CTRL, CAPT and EAM-DART states.

## 1. Settings

In [ ]:
from pathlib import Path

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
OBSERVATION_ROOT = Path("/compyfs/zhan391/acme_init/Observations")  # observations (shared archive)
workflow_name = "fcst_atm_timeseries"

run_segment = "fc"
# Forecast cases: season -> experiments compared together (same period).
cases = {
    "S0": ("CTRL10-S0", "CAPT10-S0", "DART20-S0", "DART40-S0"),  # 201201-201202
    "S1": ("CTRL10-S1", "CAPT10-S1", "DART40-S1"),  # 201206-201207
}
variables = ("PRECT", "FLUT")
regions = ("global", "CONUS", "NHMidLat", "Tropics")
max_members = None  # e.g. 2 for a quick test
# Existing caches are reused (they are keyed by season, not by experiment
# list); set force_compute after changing a case's experiments.
force_compute = False
show = True
FIGURE_DPI = 200

## 2. Project setup

In [ ]:
import sys
import matplotlib.pyplot as plt

repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.bias_metrics import experiment_registry
from util.forecast_regional_timeseries import REGIONS, cache_regional_timeseries, plot_regional_timeseries, timeseries_cache_path

data_dir, figure_dir = workflow_output_dirs(
    workflow_name, output_root=DIAGNOSTIC_OUTPUT_ROOT
)
unknown = set(variables) - {"PRECT", "FLUT"}
if unknown:
    raise ValueError(f"Observation-comparison time series support PRECT and FLUT, not {sorted(unknown)}")
unknown = set(regions) - set(REGIONS)
if unknown:
    raise ValueError(f"Unknown regions {sorted(unknown)}; choose from {list(REGIONS)}")

## 3. Compute or reuse regional time series

In [ ]:
registries, processed = {}, {}
for season, experiments in cases.items():
    registry = registries[season] = experiment_registry(INPUT_DATA_ROOT, season, run_segment)
    periods = {registry[experiment]["period"] for experiment in experiments}
    if len(periods) != 1:
        raise ValueError(f"{season}: experiments have different periods {periods}")
    period = periods.pop()
    for variable in variables:
        for region in regions:
            target = timeseries_cache_path(data_dir, variable, region, season, run_segment, period)
            existed = target.is_file()
            processed[(season, variable, region)] = cache_regional_timeseries(
                target, data_root=INPUT_DATA_ROOT, reference_root=OBSERVATION_ROOT,
                experiments=experiments, season=season, run_segment=run_segment,
                variable=variable, region=region, period=period,
                max_members=max_members, force_compute=force_compute,
            )
            print("[REUSE]" if existed and not force_compute else "[WRITE]", target)

## 4. Plot, save, and show

In [ ]:
for (season, variable, region), ds in processed.items():
    registry = registries[season]
    fig = plot_regional_timeseries(
        ds, label_map={name: registry[name]["label"] for name in cases[season]}
    )
    output = figure_dir / f"{variable}_{region}_{season}_{run_segment}_timeseries.png"
    fig.savefig(output, dpi=FIGURE_DPI, bbox_inches="tight")
    print("[FIGURE]", output)
    if not show:
        plt.close(fig)

## Notes: variables and regions

Variables: `PRECT` verified against GPM/IMERG and `FLUT` verified against NOAA interpolated OLR. Regions: `global`, `CONUS`, `NHMidLat`, `Tropics`, `NINO3.4`, and `MAR_CONT`.

Caches: `<variable>_<region>_<season>_fc_<period>_regional_timeseries.nc`, holding every member, the ensemble mean, the 25th/75th-percentile envelope, and the aligned observation.